In [1]:
!pip install openai

In [1]:

from openai import OpenAI
import json

api_key_detailed= 'sk-tDKrXNJk2yPn7rjhhAAW62nuyygbI0utlLZlmmSr8Msy3XT1'
# sk-tDKrXNJk2yPn7rjhhAAW62nuyygbI0utlLZlmmSr8Msy3XT1

from openai import OpenAI
import json

client = OpenAI(
    base_url="https://api.vectorengine.ai/v1",
    api_key=api_key_detailed
)


In [2]:
def get_current_weather(location, unit="fahrenheit"):
    """获取指定位置的当前天气"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

In [3]:
def run_conversation():
    # 1. 设置对话和可用函数
    messages = [{"role": "user", "content": "What's the weather like in San Francisco, Tokyo, and Paris?"}]
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather in a given location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                    },
                    "required": ["location"],
                },
            },
        }
    ]

    # 2. 创建对话
    response = client.chat.completions.create(
        model="gemini-3.1-flash-lite-preview",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    
    response_message = response.choices[0].message
    print(response_message)
    
    # 3. 处理函数调用
    tool_calls = response_message.tool_calls
    if tool_calls:
        available_functions = {
            "get_current_weather": get_current_weather,
        }
        
        messages.append(response_message)
        
        # 4. 执行函数调用并获取结果
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            
            function_response = function_to_call(
                location=function_args.get("location"),
                unit=function_args.get("unit"),
            )
            
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            })
        
        # 5. 获取最终响应
        second_response = client.chat.completions.create(
            model="gemini-3.1-flash-lite-preview",
            messages=messages,
        )
        return second_response

# 执行对话
print(run_conversation())

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_a71d8942a03d477d8ce048ead1ee3498', function=Function(arguments='{"location":"San Francisco, CA"}', name='get_current_weather'), type='function', extra_content={'google': {'thought_signature': 'EjQKMgEMOdbHo07JKuWhRKS1RVcF7GXmol4nXZWi7obMgv9GNjHkRAO+K8OdvDthJDC2Nczd'}}), ChatCompletionMessageFunctionToolCall(id='call_578be0d548754bf89e4bc21d826366ba', function=Function(arguments='{"location":"Tokyo, Japan"}', name='get_current_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_ac73d1f1a2b448d5b0eda8679c3f5370', function=Function(arguments='{"location":"Paris, France"}', name='get_current_weather'), type='function')])
ChatCompletion(id='chatcmpl-20260422005654138180150wBT536hY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here is the c